# Daily LightGBM feature ablation on Kaggle CPU

Trains the daily LightGBM of EarlyMule-TG from the exported dataset `80K-earlymule-mule` with feature families or windows left out (`python -m earlymule.models.lightgbm_export`, ablations in `configs/lightgbm_ablation.yaml`). CPU only, so it does not use the weekly GPU quota. The (ablation, seed) pairs in `JOBS` run one after another, each with LightGBM's 4 threads; each run writes validation and test scores, `importance.csv`, `model.txt` and `run_meta.json` to `/kaggle/working/lgbm_abl_<ablation>/seed<N>/` as soon as it ends, so a session stopped at its 12-hour limit keeps the finished runs. Evaluation runs locally with `python -m earlymule.evaluation.import_scores`.

Input: the private dataset `verzol/earlymule-80k-earlymule-mule` (made by `python -m earlymule.data.export_events`). One notebook per batch is generated by `notebooks/kaggle/lgbm_ablation/make_batches.py`, which rewrites the parameter cell.

In [ ]:
# Run parameters: (ablation, seed) pairs, run one after another; commit to check out (None = latest main)
JOBS = [('full', 0), ('full', 1), ('full', 2), ('no_churn', 0), ('no_churn', 1), ('no_churn', 2)]
COMMIT = None

In [ ]:
# Clone the repository at COMMIT and install it without dependencies (LightGBM, pandas and PyYAML are preinstalled)
!git clone -q https://github.com/Verzol/earlymule-tg.git /tmp/earlymule-tg
if COMMIT:
    !git -C /tmp/earlymule-tg checkout -q {COMMIT}
!git -C /tmp/earlymule-tg log -1 --format='%h %s'
!pip install -q -e /tmp/earlymule-tg --no-deps
!python -c "import lightgbm, pandas; print('lightgbm', lightgbm.__version__, 'pandas', pandas.__version__)"
!nproc; free -g

The dataset folder is found by its manifest, wherever Kaggle mounts it. Each run is its own process, so its memory is released before the next one starts; logs go next to the outputs.

In [ ]:
# Train every (ablation, seed) pair in order; a failed run is logged and the next one still starts
import json, subprocess, time
from pathlib import Path
data = next(m.parent for m in Path('/kaggle/input').rglob('manifest.json')
            if json.loads(m.read_text())['dataset'] == '80K-earlymule-mule')
codes = []
for ablation, seed in JOBS:
    out = Path('/kaggle/working') / f'lgbm_abl_{ablation}' / f'seed{seed}'
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    cmd = ['python', '-m', 'earlymule.models.lightgbm_export', '--data', str(data), '--ablation', ablation,
           '--seed', str(seed), '--out', str(out)]
    code = subprocess.call(cmd, cwd='/tmp/earlymule-tg', stdout=open(out / 'run.log', 'w'), stderr=subprocess.STDOUT)
    codes.append((ablation, seed, code, round((time.time() - t0) / 60, 1)))
    print(codes[-1], flush=True)
print(codes)

In [ ]:
# Last lines of every run log
for log in sorted(Path('/kaggle/working').glob('lgbm_abl_*/seed*/run.log')):
    print(f'--- {log.parent.parent.name}/{log.parent.name}')
    print(log.read_text()[-1500:])